In [2]:
import pandas as pd
import numpy as np
import yfinance as yf

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error,
    accuracy_score, confusion_matrix,
    roc_auc_score, roc_curve
)

# GENERAL PARAMETERS

START_DATE = "2017-11-20"
END_DATE   = "2025-01-01"

WINDOW_VOL  = 30   # window for realized volatility
WINDOW_CORR = 30   # window for rolling correlation
HORIZON     = 5    # forecasting horizon (ex: predict 5 days ahead)


In [4]:
# A1 Download raw price data

tickers = ["^GSPC", "BTC-USD", "GC=F", "^VIX"]  # SP500, Bitcoin, Gold, VIX

raw = yf.download(tickers, start=START_DATE, end=END_DATE)["Close"]
raw.columns = ["SP500", "BTC", "Gold", "VIX"]


/tmp/ipykernel_1226/3297750540.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw = yf.download(tickers, start=START_DATE, end=END_DATE)["Close"]
[*********************100%***********************]  4 of 4 completed


In [6]:
# A2 — Align data on SP500 trading days

# Keep only days where SP500 is traded (SP500 = main market clock)
df = raw.copy()
df = df[df["SP500"].notna()]

# Force business day frequency
df = df.asfreq("B")

# Forward-fill assets that trade 24/7 or have missing values
df[["BTC", "Gold", "VIX"]] = df[["BTC", "Gold", "VIX"]].ffill()


In [8]:
# A3 — Compute returns, volatility, correlation

# Log returns
for col in ["SP500", "BTC", "Gold"]:
    df[f"ret_{col}"] = np.log(df[col]).diff()

# Realized volatility (rolling window)
for col in ["SP500", "BTC", "Gold"]:
    df[f"vol_{col}"] = df[f"ret_{col}"].rolling(WINDOW_VOL).std()

# Rolling correlation BTC–SP500
df["corr_BTC_SP500"] = (
    df["ret_BTC"].rolling(WINDOW_CORR).corr(df["ret_SP500"])
)

# Annualized volatility (optional)
for col in ["SP500", "BTC", "Gold"]:
    df[f"vol_ann_{col}"] = df[f"vol_{col}"] * np.sqrt(252)

# Drop initial NaN values from rolling windows
df = df.dropna()
